# Five-LLM comparison and visualizations — Colab workflow

This notebook compares the five model families **within the same scenario and controlled condition** across situational, cultural, and game cases.

**Hardware:** CPU is sufficient. No LLM inference occurs, and no API key is needed. The script reparses/evaluates saved artifacts, then writes dependency-free SVG figures.

The controlled sets are:
- Situational: `g0_r0`, 11 × 5 outputs.
- Cultural: actual location-revealed outputs, 14 × 5. Derived anonymized DeepSeek rows are excluded.
- Game: three runs, 9 × 5 × 3 outputs.


In [ ]:
from pathlib import Path
import os, subprocess

# Choose ONE setup method.
REPO_URL = ""  # e.g. https://github.com/<organization>/world-model.git
REPO_DIR = Path("/content/world-model")

if not REPO_DIR.exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL above, or upload/mount the repository at /content/world-model")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repository:", Path.cwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## Run the controlled comparison

This creates per-output, per-scenario, and per-model-pair CSVs under `results/cross_model_comparison`.


In [ ]:
subprocess.run(["python", "experiments/compare_five_model_outputs.py"], check=True)


## Generate radar, heatmap, and divergence figures

The radar axes are normalized to 0–1. Structural agreement is name-insensitive feature-multiset F1, not semantic equivalence.


In [ ]:
subprocess.run(["python", "experiments/plot_five_model_comparison.py"], check=True)


## Display headline tables


In [ ]:
import json, pandas as pd
from IPython.display import display

summary = json.loads(Path("results/cross_model_comparison/summary.json").read_text())
display(summary)
display(pd.read_csv("results/cross_model_comparison/summary_by_model.csv"))
display(pd.read_csv("results/cross_model_comparison/summary_by_model_pair.csv"))
display(pd.read_csv("results/cross_model_comparison/scenario_agreement.csv"))


## Display all visualizations


In [ ]:
from IPython.display import SVG, display
figures = sorted(Path("results/cross_model_comparison/figures").glob("*.svg"))
for figure in figures:
    print(figure.name)
    display(SVG(filename=str(figure)))


## Interpretation guardrails

- A larger radar area is **not** an overall model-quality score; read each axis separately.
- Structural F1 measures representation shape, not semantic correctness.
- Generic sensitive-term flags request human review; they are not proof of bias.
- The one case-specific forbidden-content finding is the strongest automated ethical concern and still requires human confirmation.
- Invalid outputs are excluded from pairwise structural F1, so check `comparable_scenarios` before comparing averages.


## Download the complete comparison (optional)


In [ ]:
import shutil
archive = shutil.make_archive("/content/five_model_comparison", "zip", REPO_DIR / "results/cross_model_comparison")
print("Created", archive)
from google.colab import files
files.download(archive)
